In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [24]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig


MAX_SEQ_LENGTH = 8000
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
EPOCHS = 2
LEARNING_RATE = 2e-4
SYSTEM_PROMPT = """You are a medical assistant AI trained to identify possible diseases based on given symptoms.

You will be provided with a list of symptoms as input. Your task is to:
1. Predict the most likely disease.
2. Suggest appropriate precautions or basic treatments based on the prediction."""
DATASET_PATH = "combined_disease_dataset.csv"
OUTPUT_DIR = "outputs"
load_in_4bit = False


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = load_in_4bit,
    full_finetuning=False
)


dataset = load_dataset("csv", data_files=DATASET_PATH)["train"]


def format_example(example):
    precautions = example["precautions"] if example["precautions"] else "Consult a doctor for proper diagnosis."

    assistant_reply = f"""
POSSIBLE DISEASE: {example["disease"]}

POSSIBLE PRECAUTIONS: {precautions}"""
    return {
        "text": (
            f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
            f"<|im_start|>user\n{example['symptoms']}\n<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_reply}\n<|im_end|>"
        )
    }


dataset = dataset.map(format_example)
dataset = dataset.shuffle(seed=42)
print ("Dataset loaded properly")
print (f"First row of the dataset:\n {dataset[0]["text"]}")


model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 64,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)


FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",          
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=30,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        remove_unused_columns=False,
    ),
)


trainer_stats = trainer.train()
print("\n✅ Training complete!")
print(f"   Runtime : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Loss    : {trainer_stats.metrics['train_loss']:.4f}")


==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

HuggingFaceTB/SmolLM2-360M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.
Dataset loaded properly
First row of the dataset:
 <|im_start|>system
You are a medical assistant AI trained to identify possible diseases based on given symptoms.

You will be provided with a list of symptoms as input. Your task is to:
1. Predict the most likely disease.
2. Suggest appropriate precautions or basic treatments based on the prediction.
<|im_end|>
<|im_start|>user
vomiting,headache,nausea,spinning movements,loss of balance,unsteadiness
<|im_end|>
<|im_start|>assistant

POSSIBLE DISEASE: (vertigo) Paroymsal  Positional Vertigo

POSSIBLE PRECAUTIONS: lie down, avoid sudden change in body, avoid abrupt head movment, relax
<|im_end|>
Unsloth: We found double BOS tokens - we shall remove one automatically.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,325 | Num Epochs = 2 | Total steps = 666
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 17,367,040 of 379,188,160 (4.58% trained)


Step,Training Loss
1,2.699224
2,2.739916
3,2.673545
4,2.563898
5,2.593741
6,2.397973
7,2.211983
8,2.030586
9,1.864404
10,1.765061


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-666/tokenizer_config.json.



✅ Training complete!
   Runtime : 2320s
   Loss    : 0.1514


In [25]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 960, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=960, out_features=960, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=960, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=960, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lo

In [26]:
def generate_response(symptoms):
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{symptoms}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [27]:
test_symptoms = [
    "fever, cough, sore throat, runny nose",
    "fever, cough, shortness of breath, fatigue", 
    "headache, nausea, vomiting, sensitivity to light",
    "fever, rash, body pain, joint pain",
    "abdominal pain, diarrhea, vomiting",
    "burning urination, frequent urination, lower abdominal pain",
    "chest pain, shortness of breath, sweating",
    "sneezing, runny nose, itchy eyes",
    "polyuria, excessive thirst, fatigue, weight loss",
    "high fever, chills, headache, muscle pain",
    "skin itching, red patches, dry skin",
    "sore throat, fever, swollen tonsils",
    "abdominal pain, bloating, constipation",
    "fever, headache, stiff neck, nausea",
    "cough, fever, night sweats, weight loss",
    "My head is pounding, I feel like throwing up, and bright light bothers me"
]

# Expected results (for validation):
expected_results = [
    "Common cold / flu-like illness",
    "Respiratory infection",
    "Migraine", 
    "Viral infection / dengue-like illness",
    "Gastroenteritis",
    "Urinary tract infection",
    "Cardiac emergency",
    "Allergic rhinitis",
    "Diabetes mellitus",
    "Malaria-like febrile illness",
    "Skin allergy / dermatitis",
    "Tonsillitis",
    "Digestive disorder / constipation",
    "Meningitis warning signs",
    "Tuberculosis-like illness",
    "Migraine"
]

# Usage example:
for symptoms, expected in zip(test_symptoms, expected_results):
    result = generate_response(symptoms)
    print(f"Input: {symptoms}")
    print(f"Expected: {expected}")
    print(f"Got: {result}")
    print("---")

Input: fever, cough, sore throat, runny nose
Expected: Common cold / flu-like illness
Got: system
You are a medical assistant AI trained to identify possible diseases based on given symptoms.

You will be provided with a list of symptoms as input. Your task is to:
1. Predict the most likely disease.
2. Suggest appropriate precautions or basic treatments based on the prediction.

user
fever, cough, sore throat, runny nose

assistant

POSSIBLE DISEASE: Common Cold

POSSIBLE PRECAUTIONS: drink vitamin c rich drinks, take vapour, avoid cold food, keep fever in check

---
Input: fever, cough, shortness of breath, fatigue
Expected: Respiratory infection
Got: system
You are a medical assistant AI trained to identify possible diseases based on given symptoms.

You will be provided with a list of symptoms as input. Your task is to:
1. Predict the most likely disease.
2. Suggest appropriate precautions or basic treatments based on the prediction.

user
fever, cough, shortness of breath, fatigue


In [32]:
original_test_symptoms = [
    "fever, cough, sore throat, runny nose",
    "fever, cough, shortness of breath, fatigue",
    "headache, nausea, vomiting, sensitivity to light",
    "fever, rash, body pain, joint pain",
    "abdominal pain, diarrhea, vomiting",
    "burning urination, frequent urination, lower abdominal pain",
    "chest pain, shortness of breath, sweating",
    "sneezing, runny nose, itchy eyes",
    "polyuria, excessive thirst, fatigue, weight loss",
    "high fever, chills, headache, muscle pain",
    "skin itching, red patches, dry skin",
    "sore throat, fever, swollen tonsils",
    "abdominal pain, bloating, constipation",
    "fever, headache, stiff neck, nausea",
    "cough, fever, night sweats, weight loss",
    "My head is pounding, I feel like throwing up, and bright light bothers me"
]

original_expected = [
    "Common cold / flu-like illness",
    "Respiratory infection",
    "Migraine",
    "Viral infection / dengue-like illness",
    "Gastroenteritis",
    "Urinary tract infection",
    "Cardiac emergency",
    "Allergic rhinitis",
    "Diabetes mellitus",
    "Malaria-like febrile illness",
    "Skin allergy / dermatitis",
    "Tonsillitis",
    "Digestive disorder / constipation",
    "Meningitis warning signs",
    "Tuberculosis-like illness",
    "Migraine"
]

new_test_symptoms = [
    "fatigue, weight gain, depression, cold intolerance",
    "yellow skin, yellow eyes, dark urine, abdominal pain",
    "severe abdominal pain, vomiting, fever",
    "painful swallowing, white throat patches, fever",
    "sudden vision loss, headache, jaw pain",
    "leg swelling, shortness breath, chest pain",
    "memory loss, confusion, difficulty speaking",
    "bloody cough, chest pain, weight loss",
    "joint swelling, morning stiffness, fatigue",
    "excessive hunger, blurred vision, slow healing",
    "rapid heartbeat, sweating, tremors, weight loss",
    "painful red eye, vision changes, headache",
    "severe back pain, fever, urinary symptoms",
    "hives, swelling lips, difficulty breathing",
    "chronic cough, wheezing, chest tightness",
    "irregular heartbeat, dizziness, fainting",
    "painful mouth sores, fever, swollen glands",
    "bloody stool, abdominal pain, weight loss",
    "sudden severe headache, vomiting, confusion",
    "dry cough, fever, loss taste smell"
]

new_expected = [
    "Hypothyroidism",
    "Jaundice/Hepatitis",
    "Appendicitis",
    "Oral thrush/Candidiasis",
    "Temporal arteritis",
    "Pulmonary embolism",
    "Stroke warning",
    "Lung cancer suspicion",
    "Rheumatoid arthritis",
    "Diabetes complication",
    "Hyperthyroidism",
    "Acute glaucoma",
    "Kidney infection/Pyelonephritis",
    "Anaphylaxis",
    "Asthma",
    "Arrhythmia",
    "Herpes simplex",
    "Colorectal issue",
    "Subarachnoid hemorrhage",
    "COVID-19 like"
]




all_symptoms = original_test_symptoms + new_test_symptoms
all_expected = original_expected + new_expected

# Usage example:
for symptoms, expected in zip(all_symptoms, all_expected):
    result = generate_response(symptoms)
    print(f"Input: {symptoms}")
    print(f"Expected: {expected}")
    print(f"Got: {result}")
    print("---")

Input: fever, cough, sore throat, runny nose
Expected: Common cold / flu-like illness
Got: system
You are a medical assistant AI trained to identify possible diseases based on given symptoms.

You will be provided with a list of symptoms as input. Your task is to:
1. Predict the most likely disease.
2. Suggest appropriate precautions or basic treatments based on the prediction.

user
fever, cough, sore throat, runny nose

assistant

POSSIBLE DISEASE: Common Cold

POSSIBLE PRECAUTIONS: drink vitamin c rich drinks, take vapour, avoid cold food, keep fever in check

---
Input: fever, cough, shortness of breath, fatigue
Expected: Respiratory infection
Got: system
You are a medical assistant AI trained to identify possible diseases based on given symptoms.

You will be provided with a list of symptoms as input. Your task is to:
1. Predict the most likely disease.
2. Suggest appropriate precautions or basic treatments based on the prediction.

user
fever, cough, shortness of breath, fatigue


In [33]:
model.save_pretrained("local_doctor-360M")
tokenizer.save_pretrained("local_doctor-360M")

Unsloth: Restored added_tokens_decoder metadata in local_doctor-360M/tokenizer_config.json.


('local_doctor-360M/tokenizer_config.json',
 'local_doctor-360M/chat_template.jinja',
 'local_doctor-360M/tokenizer.json')

In [34]:
!sudo apt-get install zip -y
!zip -r local_doctor-360M.zip local_doctor-360M

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zip
0 upgraded, 1 newly installed, 0 to remove and 44 not upgraded.
Need to get 176 kB of archives.
After this operation, 549 kB of additional disk space will be used.
Get:1 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates/main amd64 zip amd64 3.0-13ubuntu0.2 [176 kB]
Fetched 176 kB in 0s (387 kB/s)
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package zip.
(Reading database ... 142426 files and directories currently installed.)
Preparing to unpack .../zip_3.0-13ubuntu0.2_amd64.deb ...
Unpacking zip (3.0-13ubuntu0.2) ...
Setting up zip (3.0-13ubuntu0.2) ...
  adding: local_doctor-360M/ (stored 0%)
  adding: local_doctor-360M/adapter_config.json (deflated 57%)
  adding: local_doctor-360M/chat_template.jinja (deflated 42%)
  adding: local_doctor-360M/tokenizer.json (deflate